# Part 1: Neural Network Fundamentals and Training Behavior Analysis
**Dataset:** `customer_churn_nn.csv`  — Customer churn binary classification (2,000 rows)


In [ ]:
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, ConfusionMatrixDisplay, f1_score)
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Dropout, Embedding, LSTM,
                                      Conv2D, MaxPooling2D, Flatten)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

tf.random.set_seed(42)
np.random.seed(42)
print(f'TensorFlow {tf.__version__} | NumPy {np.__version__}')


---
# Part 1: Neural Network Fundamentals and Training Behavior Analysis


## Task 1: Dataset Understanding


In [ ]:
df = pd.read_csv('customer_churn_nn.csv')
print('Shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:', df.isnull().sum().sum())
df.head()

In [ ]:
print('Statistical Summary:')
display(df.describe())

print('\nTarget Variable (churn) Distribution:')
print(df['churn'].value_counts())
print(f'Churn Rate: {df["churn"].mean()*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Target distribution
churn_counts = df['churn'].value_counts()
axes[0].bar(['Retained (0)', 'Churned (1)'], churn_counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Target Variable Distribution (Churn)')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Monthly charges distribution
axes[1].hist(df['monthly_charges_inr'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Distribution of Monthly Charges (INR)')
axes[1].set_xlabel('Monthly Charges')

plt.tight_layout()
plt.show()
print('Note: Dataset is heavily imbalanced — only 1.55% churn rate.')

## Task 2: Data Preprocessing


In [ ]:
# Drop identifier column
df_proc = df.drop(columns=['customer_id']).copy()

# Encode categorical features
cat_cols = ['region', 'plan_type', 'contract_type', 'payment_method']
le = LabelEncoder()
for col in cat_cols:
    df_proc[col] = le.fit_transform(df_proc[col])
    print(f'Encoded {col}: {df_proc[col].unique()}')

print('\nNo missing values detected. Proceeding without imputation.')

In [ ]:
# Scale numerical features
X = df_proc.drop(columns=['churn']).values
y = df_proc['churn'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape}')
print(f'Testing set:  {X_test.shape}')
print(f'Features:     {X_train.shape[1]}')
print(f'Train churn rate: {y_train.mean():.3f}')

## Task 3: Neural Network Model Building


In [ ]:
# Class weights to handle imbalance
neg, pos = np.bincount(y_train)
class_weight = {0: 1.0, 1: neg / pos}
print(f'Class weight for churn=1: {class_weight[1]:.1f}x')

def build_model(hidden_layers=[64, 32], lr=0.001, dropout=0.3, activation='relu'):
    """
    Build a feed-forward neural network.
    - Input layer: automatically shaped from training data
    - Hidden layers: configurable sizes and activation
    - Output layer: sigmoid for binary classification
    """
    model = Sequential()
    model.add(Dense(hidden_layers[0], input_shape=(X_train.shape[1],), activation=activation))
    model.add(Dropout(dropout))
    for units in hidden_layers[1:]:
        model.add(Dense(units, activation=activation))
        model.add(Dropout(dropout))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Baseline model
baseline = build_model([64, 32], lr=0.001)
baseline.summary()

## Task 4: Training and Evaluation


In [ ]:
history = baseline.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weight,
    verbose=0
)
print('Training complete.')
train_loss, train_acc = baseline.evaluate(X_train, y_train, verbose=0)
test_loss,  test_acc  = baseline.evaluate(X_test,  y_test,  verbose=0)
print(f'Train Accuracy: {train_acc:.4f} | Train Loss: {train_loss:.4f}')
print(f'Test  Accuracy: {test_acc:.4f}  | Test  Loss: {test_loss:.4f}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train', color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Validation', color='tomato')
axes[0].set_title('Model Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy'); axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train', color='steelblue')
axes[1].plot(history.history['val_loss'], label='Validation', color='tomato')
axes[1].set_title('Model Loss'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss'); axes[1].legend()

plt.tight_layout()
plt.savefig('results/part1_training_curves.png', dpi=150)
plt.show()

In [ ]:
# Evaluation
y_pred = (baseline.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Retained','Churned']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned']).plot(ax=ax, colorbar=False)
ax.set_title('Confusion Matrix – Baseline Model')
plt.tight_layout()
plt.savefig('results/part1_confusion_matrix.png', dpi=150)
plt.show()

## Task 5: Hyperparameter Experimentation


In [ ]:
experiments = [
    {'name': 'Baseline (relu, 2L)',  'layers': [64, 32],       'lr': 0.001, 'act': 'relu'},
    {'name': 'Deeper (relu, 3L)',    'layers': [128, 64, 32],  'lr': 0.001, 'act': 'relu'},
    {'name': 'High LR (0.01)',       'layers': [64, 32],       'lr': 0.01,  'act': 'relu'},
    {'name': 'tanh activation',      'layers': [64, 32],       'lr': 0.001, 'act': 'tanh'},
]

results_list = []
for exp in experiments:
    m = build_model(exp['layers'], exp['lr'], activation=exp['act'])
    m.fit(X_train, y_train, epochs=50, batch_size=32,
          validation_split=0.1, class_weight=class_weight, verbose=0)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    yp = (m.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    f1 = f1_score(y_test, yp)
    results_list.append({'Experiment': exp['name'], 'Layers': str(exp['layers']),
                         'LR': exp['lr'], 'Activation': exp['act'],
                         'Test Accuracy': round(acc, 4), 'F1 Score': round(f1, 4)})
    print(f"{exp['name']}: Acc={acc:.4f}, F1={f1:.4f}")

comp_df = pd.DataFrame(results_list)
comp_df.to_csv('results/model_comparison_table.csv', index=False)
display(comp_df)

In [ ]:
# Comparison chart
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(comp_df))
bars = ax.bar(x, comp_df['F1 Score'], color=['steelblue','tomato','seagreen','orange'])
ax.set_xticks(x)
ax.set_xticklabels(comp_df['Experiment'], rotation=10, ha='right')
ax.set_ylim(0, 1); ax.set_ylabel('F1 Score')
ax.set_title('Hyperparameter Experiment – F1 Score Comparison')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{bar.get_height():.4f}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('results/model_comparison_table.png', dpi=150)
plt.show()

## Task 6: Final Reflection

### Weights and Biases
Weights define how much influence each input feature has on a neuron's output. Biases allow the model to shift the activation function, enabling it to fit data that does not pass through the origin. During training, both are iteratively updated via backpropagation to minimise the loss.

### Activation Functions
Without non-linear activation functions, stacking multiple layers is equivalent to a single linear transformation — the network cannot learn complex, non-linear decision boundaries. ReLU (`max(0,x)`) is commonly chosen because it is computationally cheap and avoids the vanishing gradient problem that affects sigmoid and tanh in deep networks.

### Learning Rate Effects
- **Too high (e.g. 0.01):** The optimiser overshoots minima, causing unstable or diverging loss.
- **Too low (e.g. 0.00001):** Convergence is extremely slow and the model may get stuck in suboptimal local minima.

### Overfitting / Underfitting Observations
The dataset is severely imbalanced (~1.5% churn rate). Test accuracy of ~94% is misleading — the model largely predicts the majority class. F1 scores (0.09–0.26) reveal the true limitation. Class weighting helped partially; further improvements would require SMOTE oversampling or threshold tuning. The training vs. validation loss curves show the baseline is learning, but limited by class imbalance rather than classic overfitting.
